<a href="https://colab.research.google.com/github/anirbanghoshsbi/.github.io/blob/master/work/err/cross_section_momentum/ON%20ENTIRE%20EQUITY/cross_sectional_momentum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#from google.colab import drive
#drive.mount('/content/drive')

Mounted at /content/drive


In [19]:
import pandas as pd
import duckdb
import numpy as np

In [22]:
import pandas as pd

df = pd.read_parquet("/content/drive/MyDrive/equities_combined.parquet")



    SYMBOL SERIES   OPEN   HIGH    LOW  CLOSE   LAST  PREVCLOSE  TOTTRDQTY  \
0  0IHFL26     YR  984.0  984.0  984.0  984.0  984.0     1100.0          3   
1  0IHFL26     YR  920.0  920.0  920.0  920.0  920.0      984.0         10   
2  0IHFL26     YR  920.0  936.0  920.0  930.6  925.2      920.0         30   
3  0IHFL26     YR  920.0  920.0  920.0  920.0  920.0      930.6         10   
4  0IHFL26     YR  920.0  920.0  920.0  920.0  920.0      920.0         10   

   TOTTRDVAL  TIMESTAMP  TOTALTRADES          ISIN  
0     2952.0 2024-05-06            1  INE148I07SF0  
1     9200.0 2024-05-22            1  INE148I07SF0  
2    27766.0 2024-05-23            4  INE148I07SF0  
3     9200.0 2024-05-27            1  INE148I07SF0  
4     9200.0 2024-05-29            1  INE148I07SF0  


In [24]:
df['TIMESTAMP'].max()

Timestamp('2026-03-06 00:00:00')

In [25]:
# Convert date column
df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'], format='%Y-%m-%d', errors='coerce')

# Remove bad rows
df = df.dropna(subset=['TIMESTAMP'])

# Remove duplicate symbol-date pairs
df = df.drop_duplicates(subset=['SYMBOL', 'TIMESTAMP'])

# Sort properly
df = df.sort_values(['SYMBOL', 'TIMESTAMP'])

# 6-month momentum (approx 126 trading days)
lookback = 126
df['mom_6m'] = df.groupby('SYMBOL')['CLOSE'].pct_change(lookback)

# Remove rows where momentum cannot be calculated
df = df.dropna(subset=['mom_6m'])

# Cross-sectional ranking each day
df['percentile'] = df.groupby('TIMESTAMP')['mom_6m'].rank(pct=True)

# Assign momentum buckets
df['momentum_group'] = 'middle'
df.loc[df['percentile'] >= 0.9, 'momentum_group'] = 'winner'
df.loc[df['percentile'] <= 0.1, 'momentum_group'] = 'loser'

# Winners and losers portfolios
long_portfolio = df[df['momentum_group'] == 'winner']
short_portfolio = df[df['momentum_group'] == 'loser']

# Display sample
print(df[['TIMESTAMP','SYMBOL','CLOSE','mom_6m','percentile','momentum_group']].head())

     TIMESTAMP      SYMBOL    CLOSE    mom_6m  percentile momentum_group
141 2026-02-23    0MOFSL26  1158.00  0.216387    0.891230         middle
142 2026-03-05    0MOFSL26  1161.89  0.162006    0.882752         middle
143 2026-03-06    0MOFSL26  1159.99  0.160106    0.883768         middle
535 2025-07-02  1003IIFL29   999.20  0.052998    0.730410         middle
536 2025-07-04  1003IIFL29   995.00 -0.014851    0.582374         middle


In [26]:
latest_date = df['TIMESTAMP'].max()

In [34]:
formation_date = '2026-03-06'

winners_today = df[
    (df['momentum_group'] == 'winner') &
    (df['TIMESTAMP'] == formation_date) &
    (df['TOTTRDVAL']>5000000)
]

print(winners_today[['TIMESTAMP','SYMBOL','CLOSE','mom_6m','percentile','TOTTRDVAL']])

         TIMESTAMP      SYMBOL    CLOSE    mom_6m  percentile     TOTTRDVAL
170786  2026-03-06     ACUTAAS  2250.70  0.600669    0.977296  8.207375e+08
171427  2026-03-06  ADANIENSOL   992.50  0.300701    0.925449  1.521960e+09
192973  2026-03-06    AEROFLEX   220.24  0.261253    0.916300  2.414017e+08
194778  2026-03-06      AETHER  1011.95  0.356592    0.937309  7.997529e+08
223155  2026-03-06  AJANTPHARM  2992.00  0.206452    0.901050  2.566242e+08
...            ...         ...      ...       ...         ...           ...
3595275 2026-03-06        VEDL   721.00  0.670528    0.980346  9.707058e+09
3601124 2026-03-06    VENUSREM   704.85  0.523177    0.971535  6.995946e+06
3611463 2026-03-06       VHLTD   149.09  0.270256    0.917655  9.989149e+06
3656007 2026-03-06    VMARCIND   698.95  0.471009    0.960352  3.580496e+07
3671429 2026-03-06         VTL   538.90  0.307375    0.926804  1.025771e+08

[189 rows x 6 columns]
